In [ ]:
from IPython.display import clear_output
! pip install alphagenome
clear_output()

In [ ]:
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.models import variant_scorers
from alphagenome.interpretation import ism
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from alphagenome.models import dna_client

# Must obtain API key from AlphaGenome
dna_model = dna_client.create(colab_utils.get_api_key())

In [ ]:
# The GTF file contains information on the location of all trancripts.
# Note that we use genome assembly hg38 for human.
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

# Set up transcript extractors using the information in the GTF file.
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gene_annotation.filter_to_longest_transcript(gtf_transcripts)
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

Perform in silico mutagenesis of the MAPT promoter

In [ ]:
# Extract genomic range of MAPT and resize for context interval
# Supported resize lengths are 16KB, 100KB, 500KB, and 1MB
seqeunce_interval = gene_annotation.get_gene_interval(gtf, gene_symbol='MAPT')
sequence_interval = sequence_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)
sequence_interval

Interval(chromosome='chr17', start=45370267, end=46418843, strand='.', name='')

In [ ]:
# create mutagenesis interval
# this interval encompasses a 2kb region around the MAPT TSS(1 kb in each direction)
ism_interval = genome.Interval('chr17', 45893535, 45895536)

In [ ]:
# Specify recommended RNA-seq scorer
variant_scorer = [variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']]

In [ ]:
# Run rna-seq scorer
variant_scores = dna_model.score_ism_variants(
    interval=sequence_interval,
    ism_interval=ism_interval,
    variant_scorers=variant_scorer,
)

  0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# data format
# variant_scores[variant#][scorer#].X[gene#][cell_ontology#/strand#]

In [ ]:
# define function for aggregating RNAseq data for a specific cell-type and gene
# Since there is only one scorer, the scorer parameter is set to 0 by default
def aggregate_rnaseq_data (var_data, gene, cell_type, scorer=0, strand='+'):

  # determine values for gene and cell ontology/strand
  gene_val = int(var_data[0][scorer].obs[var_data[0][scorer].obs['gene_name'] == gene].index.to_numpy()[0])
  strand_ont_val = int(var_data[0][scorer].var[(var_data[0][scorer].var['ontology_curie'] == cell_type) & (var_data[0][scorer].var['strand'] == strand)].index.to_numpy()[0])

  # create output dataframe
  output_df = pd.DataFrame({'gene': [], 'cell_type': [], 'variant_pos': [], 'ref': [], 'alt': [], 'is_alt': [], 'score': []})

  # iterate through var_data and extract info
  for i in range(len(var_data)):
    # extract variant position
    var_pos = var_data[i][scorer].uns['variant'].position

    #extract variant ref
    var_ref = var_data[i][scorer].uns['variant'].reference_bases

    #extract variant alt
    var_alt = var_data[i][scorer].uns['variant'].alternate_bases

    # extract score
    score = float(var_data[i][scorer].X[gene_val][strand_ont_val])

    output_df = pd.concat([output_df, pd.DataFrame({'gene': [gene], 'cell_type': [cell_type], 'variant_pos': [var_pos], 'ref': [var_ref], 'alt': [var_alt], 'is_alt': [1], 'score': [score]})], ignore_index=True)

  return output_df

In [ ]:
mapt_glut_scores = aggregate_rnaseq_data(variant_scores, 'MAPT', 'CL:0000679', 0, '+')
mapt_glut_scores.to_csv('mapt_prom_mutagenesis_scores_1MB_window.csv')

# CL:0000679 = Glutamateric Neurons
# CL:0002518 = Embryonic Kidney Epithelial Cells
# UBERON:0000955 = Brain Tissue
# CL:0000047 = Neuronal Stem Cell (embryonic)